# scTASL tutorial: multi-omics clustering

This notebook trains the scTASL multi-omics clustering model on
paired scRNA-seq and scATAC-seq data, generates one cell representation,
performs Leiden clustering, and evaluates biological structure.

**Expected inputs:** one RNA and one ATAC H5AD file with matching paired-cell
observation names. This tutorial dataset includes `obs["cell_type"]` for labeled
UMAPs and supervised evaluation; cell-type labels are not used to train the
model and may be unavailable in real applications.

> Run this notebook from the repository root.


## 1. Setup

Import the model, evaluation metrics, and plotting utilities used below.


In [ ]:
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import torch
from IPython.display import display
import warnings
warnings.filterwarnings("ignore")

from model.multi_omics_clustering_model import MultiOmicsClusteringModel
from utils.common.metrics import (
    adjusted_rand_index,
    avg_silhouette_width_label,
    compute_bcs,
    norm_lisi_label,
    normalized_mutual_information,
)
from utils.common.data_configure import load_omics_inputs
from utils.common.plots import cell_type_palette

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
sc.set_figure_params(dpi=80, facecolor="white")


### Configure preprocessing and training

The graph connects cells with selected RNA genes and ATAC peaks. Larger
`K_GENE` and `K_PEAK` values create denser graphs and increase memory use.
`EPOCHS` controls the main training cost.


In [ ]:
DATASET = "PBMC-10k"
TASK = "multi-omics_clustering"

N_HVG = 3000
N_COMPONENTS = 50
K_GENE = 500
K_PEAK = 1000

RANDOM_SEED = 0

DATA_DIR = Path("dataset") / DATASET
RNA_PATH = DATA_DIR / "10x-Multiome-Pbmc10k-RNA.h5ad"
ATAC_PATH = DATA_DIR / "10x-Multiome-Pbmc10k-RNA.h5ad"
RESULT_DIR = Path("results") / DATASET / TASK
ADATA_DIR = RESULT_DIR / "adata"
FIGURE_DIR = RESULT_DIR / "figure"
METRICS_DIR = RESULT_DIR / "metrics"

for output_dir in (RESULT_DIR, ADATA_DIR, FIGURE_DIR, METRICS_DIR):
    output_dir.mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print(f"Dataset: {DATASET}")
print(f"Results: {RESULT_DIR.resolve()}")
print(f"Compute device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")


## 2. Validate paired source files

A lightweight backed read checks file structure and paired-cell identity
before the more expensive graph-construction step.


In [ ]:
# Clustering uses raw source objects only for a quick metadata check here.
# The source files do not need a counts layer at this stage.
# This tutorial requires cell_type only for labeled plots and evaluation metrics.
# For unlabeled real data, set require_cell_type=False and skip label-based sections.
rna, atac, _, source_summary = load_omics_inputs(
    rna_path=RNA_PATH,
    atac_path=ATAC_PATH,
    backed="r",
    require_counts=False,
    require_cell_type=True,
    require_paired=True,
    paired_task="paired multi-omics clustering",
    close_backed=True,
)

display(source_summary)
print("rna:", rna, "\natac:", atac)


## 3. Preprocess omics inputs and construct the heterogeneous graph

Preprocessing selects highly variable RNA genes, computes low-dimensional
modality representations, filters ATAC peaks, and builds cell–gene and
cell–peak neighborhoods for the graph model.


In [ ]:
graph_data, rna, atac = MultiOmicsClusteringModel.data_processing(
    rna_path=str(RNA_PATH),
    atac_path=str(ATAC_PATH),
    n_hvg=N_HVG,
    n_comp=N_COMPONENTS,
    k_gene=K_GENE,
    k_peak=K_PEAK,
)

# The processed RNA object keeps cell_type only for labeled plots and metrics.
# If your data are unlabeled, skip the label-based evaluation cells below.
if "cell_type" not in rna.obs:
    raise KeyError('Processed RNA AnnData is missing obs["cell_type"].')
rna.obs["cell_type"] = rna.obs["cell_type"].astype("category")

graph_summary = pd.Series(
    {
        "processed RNA shape": rna.shape,
        "processed ATAC shape": atac.shape,
        "graph node types": list(graph_data.node_types),
        "graph edge types": len(graph_data.edge_types),
    },
    name="value",
)
display(graph_summary.to_frame())


## 4. Initialize and train the model

The model uses contrastive objectives across RNA, ATAC, and full graph views.
It saves its configuration, training log, best checkpoint, and best projected
cell embedding under `RESULT_DIR`.


In [ ]:
model = MultiOmicsClusteringModel(
    data=graph_data,
    result_path=str(RESULT_DIR),
    seed=RANDOM_SEED,
    print_every=10,
)
training_log = model.train_model()
display(training_log.tail())

## 5. Encode cells and perform clustering

`X_hgt_hidden` stores the learned hidden cell representation. Neighbors, UMAP,
and Leiden clusters are computed from this representation rather than from
the original RNA count matrix.


In [ ]:
EMBEDDING_KEY = "X_hgt_hidden"
rna = model.encode_data(rna, embedding_key=EMBEDDING_KEY)

if rna.obsm[EMBEDDING_KEY].shape[0] != rna.n_obs:
    raise ValueError("The encoded cell count does not match the RNA AnnData.")

sc.pp.neighbors(rna, use_rep=EMBEDDING_KEY, metric="cosine")
sc.tl.umap(rna, random_state=RANDOM_SEED)
sc.tl.leiden(
    rna,
    resolution=3,
    random_state=RANDOM_SEED,
    flavor="igraph",
    n_iterations=2,
    directed=False,
)

print(f"Embedding shape: {rna.obsm[EMBEDDING_KEY].shape}")
print(f"Leiden clusters: {rna.obs['leiden'].nunique()}")


## 6. Visualize cell types and Leiden clusters

Comparing the two UMAPs shows how unsupervised clusters correspond to known
biological labels. UMAP is a visualization and should not replace the
quantitative metrics calculated below.


In [ ]:
plot_specs = {
    "cell_type": ("Original cell types", cell_type_palette(rna)),
    "leiden": ("Leiden clusters", None),
}
for color_key, (title, palette) in plot_specs.items():
    figure = sc.pl.umap(
        rna,
        color=color_key,
        palette=palette,
        title=title,
        show=False,
        return_fig=True,
    )
    figure_path = FIGURE_DIR / f"umap_{color_key}.pdf"
    figure.savefig(figure_path, bbox_inches="tight")
    display(figure)
    plt.close(figure)


## 7. Evaluation



In [ ]:
embedding = np.asarray(rna.obsm[EMBEDDING_KEY])
cell_type_labels = rna.obs["cell_type"]

metrics = {
    "NMI": normalized_mutual_information(embedding, cell_type_labels),
    "ARI": adjusted_rand_index(embedding, cell_type_labels),
    "ASW label": avg_silhouette_width_label(embedding, cell_type_labels),
    "BC": compute_bcs(embedding, cell_type_labels),
    "cLISI": norm_lisi_label(embedding, cell_type_labels),
}
metrics["Mean score"] = float(
    np.mean(list(metrics.values()))
)

metrics_table = pd.DataFrame(
    {"metric": list(metrics), "score": list(metrics.values())}
)
display(metrics_table.style.format({"score": "{:.4f}"}))
